# HealthTrack – Visualization & Insights
**Mini Project – Data Science | Dept. of Computer Science | 2024–25**

This notebook generates all output charts used in the report and web application:
- Feature importance from trained model
- Confusion matrix
- Readmission rates by category
- Numerical feature distributions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=0.95)
BLUE = '#4a7fa5'
RED  = '#b94a48'
GRAPHS_DIR = Path('../outputs/graphs')
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## 1. Load Data & Model

In [ ]:
df  = pd.read_csv('../dataset/raw_data/patient_data.csv')
clf = joblib.load('../models/rf_model.pkl')
with open('../models/model_meta.json') as f:
    meta = json.load(f)

print(f'Dataset: {df.shape}')
print(f'Model accuracy: {meta["accuracy"]*100:.1f}%')
print(f'ROC-AUC: {meta["roc_auc"]}')

## 2. Feature Importance

In [ ]:
nice = {
    'num_prior_visits':   'Prior Visits',
    'age':                'Age',
    'length_of_stay':     'Length of Stay',
    'num_medications':    'No. of Medications',
    'num_comorbidities':  'No. of Comorbidities',
    'diagnosis_enc':      'Diagnosis',
    'discharge_type_enc': 'Discharge Type',
    'insurance_enc':      'Insurance',
    'gender_enc':         'Gender',
}

fi    = meta['feature_importance']
items = sorted(fi.items(), key=lambda x: x[1])
labels = [nice.get(k, k) for k, _ in items]
values = [v for _, v in items]

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(labels, values, color=BLUE, edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance (Random Forest)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for i, v in enumerate(values):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=8, color='#888')
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'feature_importance.png', dpi=150)
plt.show()
print('Saved: feature_importance.png')

## 3. Confusion Matrix

In [ ]:
cm = np.array(meta['confusion_matrix'])

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Readmitted', 'Readmitted'],
            yticklabels=['Not Readmitted', 'Readmitted'],
            ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted', fontsize=9)
ax.set_ylabel('Actual', fontsize=9)
ax.set_title('Confusion Matrix (Test Set)', fontsize=10)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

## 4. Discharge Type vs Readmission Rate

In [ ]:
disc_rate = df.groupby('discharge_type')['readmitted_30days'].mean().sort_values(ascending=False)
print(disc_rate.round(3))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(range(len(disc_rate)), disc_rate.values * 100, color=RED, alpha=0.75, edgecolor='white')
ax.set_xticks(range(len(disc_rate)))
ax.set_xticklabels(disc_rate.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Readmission Rate (%)')
ax.set_title('Readmission Rate by Discharge Type', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'discharge_readmission.png', dpi=150)
plt.show()
print('Saved: discharge_readmission.png')

## 5. Length of Stay Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for label, val, color in [('Not Readmitted', 0, BLUE), ('Readmitted', 1, RED)]:
    ax.hist(df[df['readmitted_30days'] == val]['length_of_stay'],
            bins=18, alpha=0.6, label=label, color=color, edgecolor='white')
ax.set_xlabel('Length of Stay (days)')
ax.set_ylabel('Count')
ax.set_title('Length of Stay by Readmission Status', fontsize=11)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'los_distribution.png', dpi=150)
plt.show()
print('Saved: los_distribution.png')

## 6. Model Performance Summary

In [ ]:
print('=' * 40)
print('  Model Evaluation Summary')
print('=' * 40)
print(f'  Test Accuracy   : {meta["accuracy"]*100:.1f}%')
print(f'  ROC-AUC Score   : {meta["roc_auc"]:.4f}')
print(f'  5-Fold CV Acc.  : {meta["cv_accuracy"]*100:.1f}%')
print(f'  Training Size   : {meta["train_size"]} patients')
print(f'  Test Size       : {meta["test_size"]} patients')
print('=' * 40)

---
**End of Visualization Notebook**

All charts saved to `outputs/graphs/`